# A/B Test: Checkout Free Shipping Banner
## Stage 3b: Statistical Analysis and Product Recommendation

This notebook runs the statistical tests on the experiment results from `fct_experiment_results` and produces a go/no-go product recommendation.

**Primary metric:** Average Order Value (AOV)
**Secondary metric:** Conversion rate (order completed)
**Guardrail metric:** Return rate (order returned)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from google.cloud import bigquery

BACKGROUND = '#DDDCD6'
TEAL       = '#0D9488'
PURPLE     = '#6B21A8'
SIG_COLOR  = '#16A34A'   # green: significant
INSIG_COLOR= '#94A3B8'   # grey: not significant

ALPHA = 0.05   # significance level

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
})

In [ ]:
client = bigquery.Client(project='dbt-portfolio-498511')

df = client.query("""
    select *
    from `dbt-portfolio-498511.dbt_analytics.fct_experiment_results`
""").to_dataframe()

print(f'Rows loaded: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'Null counts:\n{df.isnull().sum()}')

In [ ]:
control   = df[df['variant'] == 'control']
treatment = df[df['variant'] == 'treatment']

print(f"Control:   {len(control):,} orders ({len(control)/len(df):.1%})")
print(f"Treatment: {len(treatment):,} orders ({len(treatment)/len(df):.1%})")

## 1. Conversion Rate: Two-Proportion Z-Test

Tests whether the proportion of completed orders differs between variants.

In [ ]:
n_control   = len(control)
n_treatment = len(treatment)
c_control   = control['is_completed'].sum()
c_treatment = treatment['is_completed'].sum()

# Conversion rate per variant (proportion of orders with is_completed = 1)
p_control   = c_control   / n_control
p_treatment = c_treatment / n_treatment

# Two-proportion z-test: tests whether the difference in conversion rates
# is statistically distinguishable from zero.
# Uses the pooled proportion under H0 (that both groups have the same rate).
z_stat, p_value_conv = proportions_ztest(
    [c_control, c_treatment],
    [n_control, n_treatment],
    alternative='two-sided'
)

# Wilson confidence intervals: more accurate than normal approximation
# for proportions near 0 or 1, or with small sample sizes.
ci_control   = proportion_confint(c_control,   n_control,   alpha=ALPHA, method='wilson')
ci_treatment = proportion_confint(c_treatment, n_treatment, alpha=ALPHA, method='wilson')

print('=== Conversion Rate ===')
print(f'Control:   {p_control:.1%}  (95% CI: {ci_control[0]:.1%} to {ci_control[1]:.1%})')
print(f'Treatment: {p_treatment:.1%}  (95% CI: {ci_treatment[0]:.1%} to {ci_treatment[1]:.1%})')
print(f'Absolute lift: {p_treatment - p_control:+.1%}')
print(f'Z-statistic: {z_stat:.4f}')
print(f'P-value: {p_value_conv:.4f}')
print(f'Significant: {p_value_conv < ALPHA}')

## 2. Average Order Value: Welch T-Test

Tests whether the mean order value differs between variants. Welch t-test is used because it does not assume equal variance between groups.

In [ ]:
control_aov   = control['order_value']
treatment_aov = treatment['order_value']

# Welch t-test: does not assume equal variance between variants.
# Returns t-statistic and two-tailed p-value.
t_stat, p_value_aov = stats.ttest_ind(
    control_aov, treatment_aov,
    equal_var=False   # Welch t-test: correct when SDs differ between groups
)

# 95% confidence interval on the difference in means.
# Uses the Welch-Satterthwaite approximation for degrees of freedom,
# which accounts for unequal variances and unequal sample sizes.
mean_diff = treatment_aov.mean() - control_aov.mean()
se_diff   = np.sqrt(
    control_aov.std()**2 / len(control_aov) +
    treatment_aov.std()**2 / len(treatment_aov)
)
df_welch = (
    (control_aov.std()**2/len(control_aov) + treatment_aov.std()**2/len(treatment_aov))**2
    / (
        (control_aov.std()**2/len(control_aov))**2 / (len(control_aov)-1) +
        (treatment_aov.std()**2/len(treatment_aov))**2 / (len(treatment_aov)-1)
    )
)
# t critical value for a two-tailed test at alpha=0.05
t_crit = stats.t.ppf(1 - ALPHA/2, df=df_welch)
ci_aov = (mean_diff - t_crit * se_diff, mean_diff + t_crit * se_diff)

print('=== Average Order Value ===')
print(f'Control mean:   £{control_aov.mean():.2f}  (SD: £{control_aov.std():.2f})')
print(f'Treatment mean: £{treatment_aov.mean():.2f}  (SD: £{treatment_aov.std():.2f})')
print(f'Absolute difference: £{mean_diff:+.2f}')
print(f'95% CI on difference: £{ci_aov[0]:.2f} to £{ci_aov[1]:.2f}')
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value_aov:.4f}')
print(f'Significant: {p_value_aov < ALPHA}')

In [ ]:
ret_control   = control['is_returned'].mean()
ret_treatment = treatment['is_returned'].mean()
ret_diff      = ret_treatment - ret_control
GUARDRAIL = 0.03

print('=== Guardrail: Return Rate ===')
print(f'Control return rate:   {ret_control:.1%}')
print(f'Treatment return rate: {ret_treatment:.1%}')
print(f'Difference: {ret_diff:+.1%}')
print(f'Guardrail breached (>{GUARDRAIL:.0%} increase): {ret_diff > GUARDRAIL}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor(BACKGROUND)

metrics = [
    ('Conversion Rate', p_control, p_treatment, ci_control, ci_treatment, p_value_conv, '{:.1%}'),
    ('Average Order Value (£)', control_aov.mean(), treatment_aov.mean(),
     (control_aov.mean() + ci_aov[0]/2, control_aov.mean() + ci_aov[1]/2),
     (treatment_aov.mean() + ci_aov[0]/2, treatment_aov.mean() + ci_aov[1]/2),
     p_value_aov, '£{:.2f}'),
]

for ax, (title, ctrl_val, treat_val, ci_ctrl, ci_treat, pval, fmt) in zip(axes, metrics):
    color = SIG_COLOR if pval < ALPHA else INSIG_COLOR
    ax.set_facecolor(BACKGROUND)
    variants = ['Control', 'Treatment']
    values   = [ctrl_val, treat_val]
    cis      = [ci_ctrl, ci_treat]
    colors   = [TEAL, PURPLE]
    ax.barh(variants, values, color=colors, height=0.5, alpha=0.85)
    for i, (val, ci) in enumerate(zip(values, cis)):
        err = abs(ci[1] - ci[0]) / 2
        ax.errorbar(val, variants[i], xerr=err, fmt='none', color='#374151', capsize=4, linewidth=1.5)
    ax.set_title(f'{title}\np = {pval:.2f} ({"significant" if pval < ALPHA else "not significant"})',
                 fontsize=11, color='#374151')
    ax.tick_params(colors='#374151')
    for spine in ax.spines.values(): spine.set_visible(False)

plt.suptitle('95% Confidence Intervals by Variant', fontsize=13, color='#374151', y=1.02)
plt.tight_layout()
plt.savefig('../images/statistical_results.png', bbox_inches='tight', facecolor=BACKGROUND)
plt.show()

## Product Recommendation

**Verdict: Inconclusive. Do not roll out.**

Neither the conversion rate lift (+0.9pp, p = 0.82) nor the AOV difference (−0.49, p = 0.96) reached statistical significance at alpha = 0.05. The confidence intervals overlap heavily and the experiment is massively underpowered: 463 total orders against the required 14,988.

At this dataset's traffic volume (~12 orders per day), detecting a 5% AOV lift would require over 1,200 days on live traffic. No guardrail breaches were observed.

**Recommended next step:** rerun on production data with a minimum of 7,500 orders per variant before making any rollout decision.